# Bronze layer

Raw ingestion into Delta tables. No transformations here: bronze keeps the data
as close to the source as possible.

## Sources

**Fingrid** (JSON API): electricity consumption and wind production, 15-minute
resolution.

**Open-Meteo** (JSON API): historical weather for 4 Finnish locations, hourly.

**ENTSO-E** (XML API): day-ahead electricity price for the Finnish bidding zone.

## Two ingestion paths

Databricks Free Edition restricts outbound internet access to an undocumented
set of allowed domains. All three source hosts were tested from this notebook:

| Host | Result |
| --- | --- |
| `archive-api.open-meteo.com` | reachable |
| `data.fingrid.fi` | DNS resolution fails |
| `web-api.tp.entsoe.eu` | DNS resolution fails |

Open-Meteo is therefore fetched directly from this notebook. Fingrid and ENTSO-E
are fetched by local scripts in `ingest/`, landed as CSV in the Unity Catalog
volume `energy_weather.landing`, and read from there.

Separating ingestion from transformation is standard practice regardless of this
constraint: production Spark clusters are frequently network isolated, with a
dedicated ingestion layer landing data first.

Open-Meteo's reachability is not guaranteed. The allowlist is not published, so
the direct path could stop working without any code change.
`ingest/fetch_weather.py` remains usable as a fallback.

## Tables

| Table | Rows | Grain |
| --- | --- | --- |
| `bronze_consumption` | 35,039 | 15 min |
| `bronze_wind` | 35,038 | 15 min |
| `bronze_weather` | 35,136 | hourly, 4 locations |
| `bronze_price` | 34,134 | mixed 60 and 15 min |

## Known issues, resolved in silver

**Inconsistent types.** `time` is a `string` in `bronze_weather`, because it is
built from JSON and JSON has no timestamp type. It is a `timestamp` elsewhere.
Column order also differs between the two ingestion paths.

**Inconsistent windows.** Fingrid was fetched with a rolling 12-month window;
weather and price use a fixed window of 2025-09-17 to 2026-09-17. The usable
overlap is roughly 2025-09-23 to 2026-09-17.

**Two price resolutions.** The European day-ahead market moved from 60-minute to
60/15-minute settlement during the covered period, so `bronze_price` contains
both `PT60M` and `PT15M` rows.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.energy_weather;

In [0]:
from ingest.fetch_weather import fetch_weather, load_weather_points

In [0]:
from datetime import datetime, timedelta, timezone

# 12 months of data, ending a couple of hours ago so we don't ask
# the API for data that doesn't exist yet
end_time = datetime.now(timezone.utc) - timedelta(hours=2)
start_time = end_time - timedelta(days=365)

start_str = start_time.strftime("%Y-%m-%dT%H:%M:%SZ")
end_str = end_time.strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Fetching from {start_str} to {end_str}")

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.energy_weather.landing;

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

CATALOG = "workspace"
SCHEMA = "energy_weather"
LANDING = "/Volumes/workspace/energy_weather/landing"

# Explicit schemas rather than inferSchema. Inference reads the file twice and can
# produce a different type on a different day, for example when a column happens to
# be entirely null in one batch. A declared schema fails loudly on a source change
# instead of silently changing the table underneath the layers above it.
FINGRID_SCHEMA = StructType([
    StructField("datasetId", IntegerType(), False),
    StructField("startTime", TimestampType(), False),
    StructField("endTime", TimestampType(), False),
    StructField("value", DoubleType(), True),
])

PRICE_SCHEMA = StructType([
    StructField("time", TimestampType(), False),
    StructField("price_eur_mwh", DoubleType(), True),
    StructField("resolution", StringType(), False),
])

# Natural key per source, which is the grain each source delivers. A key that is too
# broad matches several target rows and Delta refuses the merge; one that is too
# narrow matches nothing and inserts duplicates on every run, silently.
CSV_SOURCES = [
    ("bronze_consumption", "fingrid_consumption.csv", FINGRID_SCHEMA, ["startTime"]),
    ("bronze_wind", "fingrid_wind.csv", FINGRID_SCHEMA, ["startTime"]),
    ("bronze_price", "entsoe_price.csv", PRICE_SCHEMA, ["time"]),
]


def upsert(df: DataFrame, table: str, keys: list[str]) -> str:
    """Merge a source DataFrame into a Delta table, creating it on the first load.

    There is no WHEN NOT MATCHED BY SOURCE clause: a row the source stops
    delivering is kept rather than deleted, which is correct for an append-mostly
    time series and would be wrong for a snapshot of current state.
    """
    full_name = f"{CATALOG}.{SCHEMA}.{table}"

    if not spark.catalog.tableExists(full_name):
        df.write.format("delta").saveAsTable(full_name)
        return "initial load"

    view = f"source_{table}"
    df.createOrReplaceTempView(view)
    condition = " AND ".join(f"target.`{key}` = source.`{key}`" for key in keys)

    spark.sql(f"""
        MERGE INTO {full_name} AS target
        USING {view} AS source
            ON {condition}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    return "merged"


for table, filename, schema, keys in CSV_SOURCES:
    df = spark.read.csv(f"{LANDING}/{filename}", header=True, schema=schema)
    action = upsert(df, table, keys)
    row_count = spark.table(f"{CATALOG}.{SCHEMA}.{table}").count()
    print(f"{table}: {action}, {row_count} rows total")

In [0]:
%sql
SELECT * FROM workspace.energy_weather.bronze_consumption LIMIT 5;

In [0]:
import requests

# Open-Meteo archive API, no API key required.
# Different host than Fingrid: the host name is the only variable we are testing.
url = "https://archive-api.open-meteo.com/v1/archive"

# Smallest useful probe: one day, one location, one variable.
# We care about reachability, not the data itself.
params = {
    "latitude": 60.17,
    "longitude": 24.94,
    "start_date": "2025-01-01",
    "end_date": "2025-01-01",
    "hourly": "temperature_2m",
}

# timeout prevents the cell from hanging if packets are silently dropped
response = requests.get(url, params=params, timeout=10)

# Turns an HTTP error into an exception, so we can tell
# "connection worked but API complained" apart from "connection never happened"
response.raise_for_status()

payload = response.json()
print("Status:", response.status_code)
print("First 3 temperatures:", payload["hourly"]["temperature_2m"][:3])

In [0]:
# Repo root inside the Databricks workspace.
# Workspace files are readable with ordinary Python file I/O,
# so load_weather_points works here exactly as it does locally.
REPO_ROOT = "/Workspace/Users/dev@nikokurvinen.fi/energy-weather-lakehouse"
POINTS_CSV = f"{REPO_ROOT}/seeds/area_weather_points.csv"

points = load_weather_points(POINTS_CSV)

print("Points loaded:", len(points))
print("First point:", points[0])
print(list(points[0].keys()))

In [0]:
%sql
SELECT
    MIN(time) AS first_timestamp,
    MAX(time) AS last_timestamp,
    COUNT(*)  AS row_count
FROM workspace.energy_weather.bronze_weather

In [0]:
from datetime import date, datetime, timedelta

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, StructField, StructType, TimestampType

CATALOG = "workspace"
SCHEMA = "energy_weather"
WEATHER_TABLE = f"{CATALOG}.{SCHEMA}.bronze_weather"

# ERA5 is published daily with a five day delay, so the newest fetchable day is
# always in the past. One extra day absorbs publication jitter.
WEATHER_LAG_DAYS = 6

# Re-fetch this far behind the watermark on every run, so revised values are picked up.
RE_MERGE_DAYS = 3

# Only used on the very first load, when the table does not exist yet.
INITIAL_START = date(2025, 9, 17)

# An explicit schema instead of letting createDataFrame infer one.
# Inference sorts dict keys alphabetically, so column order changes silently.
WEATHER_SCHEMA = StructType([
    StructField("time", TimestampType(), False),
    StructField("temperature_2m", DoubleType(), True),
    StructField("wind_speed_10m", DoubleType(), True),
    StructField("area_id", StringType(), False),
    StructField("city", StringType(), False),
])


def weather_load_window():
    """Return (start, end) dates for this run, or None when there is nothing new to fetch."""
    end = date.today() - timedelta(days=WEATHER_LAG_DAYS)

    if spark.catalog.tableExists(WEATHER_TABLE):
        watermark = spark.table(WEATHER_TABLE).agg(F.max("time").alias("w")).first()["w"]
    else:
        watermark = None

    start = INITIAL_START if watermark is None else watermark.date() - timedelta(days=RE_MERGE_DAYS)

    return None if start > end else (start, end)


window = weather_load_window()
print("Load window:", window)

In [0]:
import time

if window is None:
    print("Nothing to fetch: the table is already current for the source's publication lag.")
else:
    start_date, end_date = window
    rows = []

    for point in points:
        hourly = fetch_weather(
            latitude=float(point["latitude"]),
            longitude=float(point["longitude"]),
            start_date=start_date.isoformat(),
            end_date=end_date.isoformat(),
        )

        for row in hourly:
            # Open-Meteo returns "2025-09-17T00:00" as text; parse it here so the
            # watermark comparison in the previous cell has a real timestamp to work with.
            row["time"] = datetime.fromisoformat(row["time"])
            row["area_id"] = point["area_id"]
            row["city"] = point["city"]

        rows.extend(hourly)
        print(f"{point['city']}: {len(hourly)} rows")
        time.sleep(1)

    source_df = spark.createDataFrame(rows, schema=WEATHER_SCHEMA)
    source_df.createOrReplaceTempView("weather_source")
    print("Fetched rows:", source_df.count())

In [0]:
if window is not None:
    if spark.catalog.tableExists(WEATHER_TABLE):
        spark.sql(f"""
            MERGE INTO {WEATHER_TABLE} AS target
            USING weather_source AS source
                ON  target.time = source.time
                AND target.area_id = source.area_id
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        print("Merged.")
    else:
        source_df.write.format("delta").saveAsTable(WEATHER_TABLE)
        print("Initial load written.")

    print("Rows in table:", spark.table(WEATHER_TABLE).count())